# Workshop: Delta Optimization & Performance

Practice delta optimization & performance with hands-on exercises.

| Duration | Format | Difficulty |
|---|---|---|
| 30 min | Hands-on Workshop | Intermediate |

**Prerequisites:** 02 — Optimization Demo

<!-- TRAINER-BOX -->

> **TRAINER INSTRUCTIONS**
>
> | Tier | Target audience | Time | Goal |
> |---|---|---|---|
> | **PART 1 — FUNDAMENTAL** | New to Spark / Databricks | 50 min | Building confidence, fill-in-blank |
> | **PART 2 — ADVANCED** | 1+ year experience | 65 min | Debugging, optimization, design from scratch |
>
> - **Beginner** groups: complete all PART 1, PART 2 optional
> - **Experienced** groups: PART 1 as quick review (15 min), focus on PART 2
> - **Mixed** groups: PART 1 mandatory, PART 2 for faster participants

<!-- PART1-FUNDAMENTAL -->

# PART 1 — FUNDAMENTAL (Fundamental Level)

<!-- LAB-SCENARIO -->

## Scenario

> *"Production Delta tables have grown to hundreds of millions of records and analytical queries are running slow. Your task is to perform performance tuning: file compaction, Z-ORDER clustering, and query plan verification."*

<!-- LAB-OBJECTIVES -->

## Learning Objectives

After completing this lab you will be able to:

- Use `OPTIMIZE` to compact small files and improve scan performance
- Apply `Z-ORDER BY` clustering on high-cardinality filter columns
- Use `CACHE TABLE` / `UNCACHE TABLE` for hot analytical tables
- Read and interpret `EXPLAIN` plans to identify bottlenecks
- Measure before/after performance improvements quantitatively

## Setup

In [ ]:
%run ../../setup/00_setup

In [ ]:
# Create orders table with many small files (simulating production fragmentation)
import json

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")

orders_path = f"{DATASET_PATH}/orders/orders_batch.json"
df_orders = spark.read.format("json").load(orders_path)

# Write in small batches to create many files
table_name = f"{CATALOG}.{BRONZE_SCHEMA}.orders_optimize_lab"
df_orders.repartition(20).write.mode("overwrite").saveAsTable(table_name)

# Add a few more appends to create small files
for i in range(5):
    df_orders.limit(10).write.mode("append").saveAsTable(table_name)

print(f"Table ready with fragmented files: {spark.table(table_name).count()} rows")

## Task 1: Inspect Table Metrics ~3 min

Use `DESCRIBE DETAIL` to check the number of files and total size before optimization.

**What you need to do:** Fill in the blank with `DESCRIBE DETAIL`.

**Guidance — Task 01**

The goal is to inspect table storage metrics **before** making changes — establishing a baseline.

**DESCRIBE DETAIL**
`DESCRIBE DETAIL table_name` returns a single row with Delta table metadata: `numFiles` (how many Parquet data files), `sizeInBytes` (total storage footprint), `partitionColumns`, `clusteringColumns`, and more. This is your first diagnostic tool — always check before running OPTIMIZE.

**Why file count matters**
Many small files → many file opens → slow scan performance. Each small file also adds overhead to the transaction log. The "small file problem" is the #1 reason to run OPTIMIZE.

**Things to think about**
- What's the ideal file size for a Delta table?
- How do small files get created in the first place?

In [ ]:
# TODO: Inspect table detail
df_detail = spark.sql(f"________ {table_name}")
display(df_detail.select("format", "numFiles", "sizeInBytes"))

In [ ]:
# -- Validation --
detail = df_detail.first()
num_files_before = detail["numFiles"]
assert detail["format"] == "delta", "Table should be Delta format"
print(f"Task 1 OK: {num_files_before} files, {detail['sizeInBytes']:,} bytes")

## Task 2: OPTIMIZE ~4 min

Run `OPTIMIZE` to compact the small files into larger ones.

**What you need to do:** Fill in the blank with `OPTIMIZE`. Then compare file count before/after.

**Guidance — Task 02**

The goal is to run **OPTIMIZE** — Delta's file compaction command that merges small files into larger ones.

**How OPTIMIZE works**
`OPTIMIZE table_name` reads all small Parquet files, merges them into files close to 1 GB (default target), and writes new compacted files. The old files are *not deleted* immediately — they remain for time travel until VACUUM removes them.

**Is it safe?**
Yes. OPTIMIZE is a non-destructive operation. Queries continue to work during and after optimization. The operation is idempotent — running it twice in a row does nothing the second time if no new data arrived.

**Things to think about**
- Can OPTIMIZE run on a table while a streaming job is writing to it?
- How often should you run OPTIMIZE in production — after every write?

In [ ]:
# TODO: Run OPTIMIZE
spark.sql(f"________ {table_name}")

In [ ]:
# Check files after OPTIMIZE
df_detail_after = spark.sql(f"DESCRIBE DETAIL {table_name}")
num_files_after = df_detail_after.first()["numFiles"]

print(f"Files BEFORE: {num_files_before}")
print(f"Files AFTER:  {num_files_after}")

In [ ]:
# -- Validation --
assert num_files_after <= num_files_before, "OPTIMIZE should reduce file count"
print(f"Task 2 OK: Compacted from {num_files_before} to {num_files_after} files")

## Task 3: ZORDER BY ~4 min

Run `OPTIMIZE ... ZORDER BY (customer_id)` to co-locate data for customer queries.

**What you need to do:** Fill in the blank with `ZORDER BY`.

**Guidance — Task 03**

The goal is to apply **Z-ORDER clustering** — co-locating related data within files for faster filtered queries.

**What Z-ORDER does**
Z-ORDER rearranges data within files using a space-filling curve (Z-curve). When you `ZORDER BY (customer_id)`, rows with similar `customer_id` values end up in the same files. When a query filters `WHERE customer_id = 42`, Spark can skip entire files that don't contain that value — this is called **data skipping**.

**Column selection rules**
Pick columns that appear frequently in `WHERE` clauses. Limit to 1-2 columns — more columns dilute the clustering effectiveness. Choose high-cardinality columns (many distinct values); low-cardinality columns are better served by partitioning.

**Things to think about**
- Z-ORDER is applied during OPTIMIZE — they're the same command. True or false?
- How does Z-ORDER interact with partitioning — can you use both?

In [ ]:
# TODO: OPTIMIZE with ZORDER
spark.sql(f"""
    OPTIMIZE {table_name}
    ________ (customer_id)
""")

In [ ]:
# -- Validation --
history = spark.sql(f"DESCRIBE HISTORY {table_name}").collect()
ops = [r["operation"] for r in history]
assert "OPTIMIZE" in ops, "Expected OPTIMIZE in history"
print(f"Task 3 OK: ZORDER applied. History: {ops[:5]}")

## Task 4: VACUUM ~4 min

Run `VACUUM` to remove obsolete files. Preview with `DRY RUN` first, then execute.

**What you need to do:**
1. First cell: Fill in `DRY RUN` to preview files for deletion
2. Second cell: Fill in `VACUUM` to actually remove the files

> **Warning:** We use `0 HOURS` retention for the lab. **Never** do this in production — default is 7 days.

**Guidance — Task 04**

The goal is to use **VACUUM** to permanently remove obsolete data files and reclaim storage.

**DRY RUN first**
Always preview with `VACUUM table RETAIN X HOURS DRY RUN` first. This shows which files *would be* deleted without actually deleting them. The default retention is 168 hours (7 days) — meaning files must be older than 7 days to be removed.

**Why retention matters**
Time travel relies on old files being present. Once VACUUM removes them, queries like `VERSION AS OF 5` will fail if version 5 depends on deleted files. In the lab we use `0 HOURS` for demonstration, but in production **never go below 7 days**.

**The safety check**
Databricks has a safety setting `spark.databricks.delta.retentionDurationCheck.enabled`. Setting it to `false` allows VACUUM with retention < 7 days. This is intentionally hard to do — it prevents accidental data loss.

**Things to think about**
- What happens to concurrent readers if VACUUM deletes a file they're reading?
- How does VACUUM interact with OPTIMIZE — which should run first?

In [ ]:
# Disable retention check (LAB ONLY - never do this in production!)
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

# TODO: Run VACUUM DRY RUN first
display(spark.sql(f"VACUUM {table_name} RETAIN 0 HOURS ________"))

In [ ]:
# TODO: Run actual VACUUM
spark.sql(f"________ {table_name} RETAIN 0 HOURS")
print("VACUUM complete!")

In [ ]:
# -- Validation --
# After VACUUM, old versions should no longer be accessible
current_count = spark.table(table_name).count()
assert current_count > 0, "Table should still have data"
print(f"Task 4 OK: VACUUM done. Current table: {current_count} rows")

# Re-enable safety check
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")

## Task 5: Liquid Clustering ~5 min

Create a new table with **Liquid Clustering** enabled — the modern replacement for both PARTITION BY and ZORDER.

**What you need to do:** Fill in `________ (customer_id)` → `CLUSTER BY (customer_id)`

**Guidance — Task 05**

The goal is to create a table with **Liquid Clustering** — the modern, unified replacement for both PARTITION BY and ZORDER.

**Why Liquid Clustering?**
Traditional partitioning is a design-time choice that's hard to change later. Z-ORDER requires manual OPTIMIZE runs. Liquid Clustering combines both into `CLUSTER BY (cols)` — you define clustering columns at table creation, and OPTIMIZE automatically reorganizes data. You can even change clustering columns later with `ALTER TABLE`.

**How it works**
When you `CREATE TABLE ... CLUSTER BY (customer_id)`, data is physically organized by `customer_id` using Hilbert curves (more efficient than Z-curves). Each OPTIMIZE run incrementally re-clusters only newly written data — it doesn't rewrite the entire table.

**Things to think about**
- Can you convert an existing Z-ORDERed table to Liquid Clustering?
- What happens if you `ALTER TABLE ... CLUSTER BY (product_id)` — does it recluster all existing data?

In [ ]:
# TODO: Create table with Liquid Clustering
lc_table = f"{CATALOG}.{BRONZE_SCHEMA}.orders_liquid_cluster"
spark.sql(f"DROP TABLE IF EXISTS {lc_table}")

spark.sql(f"""
    CREATE TABLE {lc_table}
    ________ (customer_id)
    AS SELECT * FROM {table_name}
""")

In [ ]:
# -- Validation --
lc_detail = spark.sql(f"DESCRIBE DETAIL {lc_table}").first()
lc_count = spark.table(lc_table).count()
assert lc_count > 0, "Liquid Clustered table should have data"
print(f"Task 5 OK: Liquid Clustered table created with {lc_count} rows")
print(f"  Clustering columns: {lc_detail['clusteringColumns']}")

## Task 6: Detect and Handle Data Skew ~5 min

Create a skewed dataset, detect the skew, and fix it with a **broadcast join**.

**What you need to do:**
1. Run setup cell to create skewed data (90% rows = customer 1)
2. Fill in `GROUP BY` / `ORDER BY` to detect skew
3. Fill in `broadcast(customers_lookup)` to optimize the join

**Guidance — Task 06**

The goal is to detect **data skew** and fix it with a **broadcast join** — a critical production optimization skill.

**What is data skew?**
Data skew means one partition has vastly more rows than others. For example, if 90% of orders belong to customer_id=1, the executor processing that key does all the work while others sit idle. This is visible as one slow task in the Spark UI while other tasks finish quickly.

**The broadcast join solution**
When one side of a join is small enough to fit in memory (< 10 MB by default), you can broadcast it to all executors using `broadcast(df)`. This eliminates the shuffle entirely — the large table stays put, and each executor gets a full copy of the small table. Spark's AQE (Adaptive Query Execution) often does this automatically, but `broadcast()` gives you explicit control.

**Things to think about**
- What if *both* sides of the join are large — can you still use broadcast?
- How does AQE's `skewJoin` optimization compare to manual broadcast?

In [ ]:
from pyspark.sql.functions import col, count, lit, rand, round as spark_round

# Step 1: Create a skewed sales table (90% hot key)
skew_table = f"{CATALOG}.{BRONZE_SCHEMA}.sales_skew_lab"

# 9000 rows for customer_id=1, 100 rows each for customers 2-11
df_hot = spark.range(9000).withColumn("customer_id", lit(1)).withColumn("amount", spark_round(rand() * 100, 2))
df_rest = spark.range(1000).withColumn("customer_id", (col("id") % 10 + 2).cast("int")).withColumn("amount", spark_round(rand() * 100, 2))
df_skewed = df_hot.unionByName(df_rest)

df_skewed.write.mode("overwrite").saveAsTable(skew_table)
print(f"Skewed table ready: {spark.table(skew_table).count()} rows")

In [ ]:
# TODO: Detect skew — count rows per customer_id, ordered DESC
# Fill in the GROUP BY and ORDER BY

df_skew_check = spark.sql(f"""
    SELECT customer_id, ________(________) as row_count
    FROM {skew_table}
    GROUP BY ________
    ORDER BY row_count ________
""")

display(df_skew_check)

In [ ]:
# -- Validation --
top_row = df_skew_check.first()
assert top_row["customer_id"] == 1, "Customer 1 should have the most rows (hot key)"
assert top_row["row_count"] > 5000, f"Customer 1 should have 9000 rows, got {top_row['row_count']}"
print(f"Skew detected! Customer {top_row['customer_id']}: {top_row['row_count']} rows (hot key)")
print(f"Other customers: ~100 rows each")

In [ ]:
from pyspark.sql.functions import broadcast, sum as spark_sum

# Create a small lookup table (customers)
customers_lookup = spark.createDataFrame(
    [(i, f"Customer_{i}") for i in range(1, 12)],
    ["customer_id", "customer_name"]
)

# TODO: Use broadcast() to join skewed sales with small customers table
df_joined = spark.table(skew_table).join(
    ________(customers_lookup),    # hint: broadcast(customers_lookup)
    on="customer_id",
    how="left"
)

In [ ]:
# Aggregate: total amount per customer
df_result = (
    df_joined
    .groupBy("customer_id", "customer_name")
    .agg(spark_sum("amount").alias("total_amount"))
    .orderBy("total_amount", ascending=False)
)

display(df_result)

In [ ]:
# -- Validation --
assert df_result.count() > 0, "Join result should not be empty"
assert "customer_name" in df_result.columns, "customer_name should be present (from broadcast join)"
top = df_result.first()
assert top["customer_id"] == 1, "Customer 1 should have highest total (9000 rows)"
print(f"Task 6 OK: Broadcast join completed. Top customer: {top['customer_name']} = ${top['total_amount']:.2f}")
print("\nKey takeaway: broadcast() sends the small table to all executors,")
print("  avoiding shuffle of the large skewed table.")

<!-- PART2-ADVANCED -->

# PART 2 — ADVANCED (Bonus — If Time Permits)

> **For whom:** Participants with 1+ year of Spark/Databricks experience
>
> **Rules:**
> - No scaffold `= None` — write from scratch
> - Tasks described as JIRA tickets (requirements + acceptance criteria)
> - Complete **at least 2 of 4 challenges**
> - Time per challenge: 8-15 minutes

### Performance — Read the Plan and Replace the Join

The following query is slow — it uses SortMergeJoin instead of BroadcastHashJoin.

```python
df_orders     = spark.table(f'{CATALOG}.{SILVER_SCHEMA}.orders')    # ~50M rows
df_products = spark.table(f'{CATALOG}.{BRONZE_SCHEMA}.products') # ~500 rows

df_result = df_orders.join(df_products, 'product_id', 'left')
df_result.explain('formatted')
```

1. Run and read the plan — find `SortMergeJoin`
2. Optimize to `BroadcastHashJoin`
3. Compare plans before/after

**Acceptance criteria:**
- `explain('formatted')` after optimization shows `BroadcastHashJoin`
- Explanation: why `df_products` is suitable for broadcast
- Configuration `autoBroadcastJoinThreshold` — when Spark does this automatically

In [ ]:
# YOUR SOLUTION — Read the Plan and Replace the Join
# ------------------------------------------------------------



### Design From Scratch — Optimization Strategy — 10B Rows

You are designing a `fact_transactions` table for RetailHub:

**Characteristics:**
- 10 billion rows, 5M daily growth
- Typical query: `WHERE transaction_date BETWEEN ... AND ...`
- Typical query: `WHERE region = 'EU' AND product_category = 'Electronics'`
- Frequent: `UPDATE ... WHERE transaction_id = ?` (corrections)

Design a strategy: partitioning, clustering, deletion vectors, OPTIMIZE schedule.

**Acceptance criteria:**
- Decision: partition by + justification (low vs high cardinality)
- Decision: ZORDER BY or Liquid Clustering — why specifically
- Decision: Deletion Vectors ON/OFF — why
- SQL DDL creating the table with full configuration
- Estimated OPTIMIZE schedule (how often, when)

In [ ]:
# YOUR SOLUTION — Optimization Strategy — 10B Rows
# ------------------------------------------------------------



### Edge Case — When Liquid Clustering Does NOT Help

Verify empirically when `CLUSTER BY` and `OPTIMIZE` do not improve performance.

**Hypothesis to test:**
If you filter by a very low cardinality column (e.g., `status IN ('active', 'inactive')`),
does Liquid Clustering on that column help?

Demonstrate on data, compare `DESCRIBE DETAIL` and scan time.

**Acceptance criteria:**
- Test table with a low-cardinality column (2-3 values)
- Scan measurement before and after OPTIMIZE
- Conclusion: what minimum cardinality makes sense for clustering
- Alternative for low-cardinality: classic partitioning

In [ ]:
# YOUR SOLUTION — When Liquid Clustering Does NOT Help
# ------------------------------------------------------------



### Refactor — Fix the ZORDER Strategy

A colleague set ZORDER on 5 columns. This is an anti-pattern.

```sql
OPTIMIZE silver_orders ZORDER BY (customer_id, product_id, order_date, region, status);
```

1. Explain why 5 columns is a problem
2. Propose and implement a better strategy (ZORDER or Liquid Clustering)
3. Justify column selection based on typical query patterns

**Acceptance criteria:**
- Written explanation of degradation with >3-4 ZORDER columns
- Fixed command with max 2 ZORDER columns or switch to CLUSTER BY
- Justified column selection based on query patterns from the description

In [ ]:
# YOUR SOLUTION — Fix the ZORDER Strategy
# ------------------------------------------------------------



## Part 2 Summary

You have completed the Advanced section for **Delta Optimization & Performance**.

**Reflection (optional):**
- Which challenge was the hardest and why?
- What would you change in your implementation?
- What approach would you use in production?

## Cleanup

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {table_name}")
spark.sql(f"DROP TABLE IF EXISTS {lc_table}")
spark.sql(f"DROP TABLE IF EXISTS {skew_table}")
print("Lab tables cleaned up")

## Lab Complete!

You have:
- Inspected table metrics with DESCRIBE DETAIL
- Compacted small files with OPTIMIZE
- Applied Z-ORDER for query optimization
- Cleaned obsolete files with VACUUM
- Created a Liquid Clustered table
- Detected data skew and resolved it with broadcast join

> **Pro Tip:** Liquid Clustering replaces both partitioning and Z-ORDER. Use `ALTER TABLE ... CLUSTER BY (new_cols)` to change clustering columns without rewriting data. For data skew, `broadcast()` works when one side is small (< 10MB by default). AQE handles skew automatically in most cases.

> **Next:** [02 -- Security & Governance Workshop](02_security_governance_workshop.ipynb)

← [02 — Optimization & Maintenance](../Demo/02_optimization_demo.ipynb) | **[ README](../../../README.md)** | [03 — Cost Management](../Demo/03_cost_management_demo.ipynb) →